In [354]:
import pandas as pd
import os
import re
from pprint import pprint

In [355]:
# load data
sas_db = pd.read_csv(f'../data/ild_schema.csv')  
# digital_db = pd.read_csv(f'../data/digital_schema.csv') 

In [356]:
def collect_files(paths):
    out = []
    for p in paths:
        path = Path(p)
        if path.is_dir():
            out.extend(sorted(path.rglob("*.sas")))
        elif path.is_file():
            out.append(path)
    return sorted(list(set(out)))

def strip_comments(text):
    # strip block comment and statement comments
    clean = re.sub(r'/\*.*?\*/', ' ', text, flags=re.S)
    clean = re.sub(r'(?m)^\s*\*[^;]*?;', ' ', clean)
    return clean

# def extract_librefs(text):
#     libname_re=re.compile(r'(?is)\blibname\s+([A-Za-z_][A-Za-z0-9_]*)\b')
#     librefs={m.group(1).lower() for m in libname_re.finditer(text)}
#     return librefs

def extract_librefs_with_name(text):
    libname_re=re.compile(r'(?is)\blibname\s+([A-Za-z_][A-Za-z0-9_]*)\b[^;\'"]*(["\'])((?:(?!\2).|\2\2)*?)\2')
    librefs=[(lib.lower(), val.replace(q*2,q).lower()) for lib,q,val in libname_re.findall(text)]
    return librefs

def iter_proc_sql(text):
    blocks= []
    proc_re= re.compile(r'(?is)\bproc\s+sql\b.*?;')
    quit_re= re.compile(r'(?is)\bquit\b\s*;')
    start = 0
    block_idx = 0
    while True:
        m = proc_re.search(text, start)
        if not m:
            break
        q = quit_re.search(text, m.end())
        if not q:
            # no terminating QUIT; consume to end
            block = text[m.end():]
            blocks.append((block_idx, block))
            break
        blocks.append((block_idx, text[m.end():q.start()]))
        block_idx += 1
        start = q.end()
    return blocks

def get_table_name_from_sql_blocks(blocks):
    # table_re=re.compile(r'(?is)\b(?:from|join)\s+([A-Za-z_][A-Za-z0-9_]*)\s*\.\s*([A-Za-z_][A-Za-z0-9_]*)\b')
    table_re=re.compile(r'(?is)(?:\b(?:from|join)\b|,)\s*([A-Za-z_][A-Za-z0-9_]*)\s*\.\s*([A-Za-z_][A-Za-z0-9_]*)\b')
    out=[]
    seen=set()
    for _, block in blocks:
        for lib, tbl in table_re.findall(block):
            key = (lib.lower(), tbl.lower())
            if key not in seen:
                seen.add(key)
                out.append((lib,tbl))
    return out

    

In [357]:
ild_for_awarding_sas_code_path=r"C:\Users\mought\OneDrive - Cambridge\Zdrive\My Projects - Git\item-level-data\SASCode"
# ild_post_awarding_sas_code_path=r"C:\Users\mought\OneDrive - Cambridge\Zdrive\My Projects - Git\item-level-data\PostAwardingILD"
# ild_step_1_sas_code_path=r"C:\Users\mought\OneDrive - Cambridge\Zdrive\My Projects - Git\item-level-data\SASCode\ild_01_identify_assessmentcomponents_requiring_recalculation.sas"
# ild_step_1_sas_code_path=r"C:\Users\mought\OneDrive - Cambridge\Zdrive\My Projects - Git\item-level-data\SASCode\ild_06_run_reports.sas"
paths=[ild_for_awarding_sas_code_path]
# paths=[ild_step_1_sas_code_path]
file_list = collect_files(paths)
all_libs= set()
all_tables=set()
for file_path in file_list:
    text=file_path.read_text()
    clean=strip_comments(text)
    libs=extract_librefs_with_name(clean)
    all_libs.update(libs)
    all_libs_alias={t[0] for t in all_libs}
    blocks=iter_proc_sql(clean)
    tables=get_table_name_from_sql_blocks(blocks)
    tables_filter = [t for t in tables if t[0] in all_libs_alias]
    all_tables.update(tables_filter)
        

In [360]:
all_libs

{('db_ca', 'ispwarehouse_candidateassessment'),
 ('db_qp', 'ispwarehouse_questionpapers'),
 ('db_r', 'ispwarehouse_research'),
 ('db_res', 'ispwarehouse_research'),
 ('db_s', 'ispwarehouse_session'),
 ('db_sd', 'ispwarehouse_staticdata'),
 ('ild', '&lib_ild'),
 ('newdir', '\\\\filestorage\\xstream\\cie\\csd\\ard_ild_reports\\&sessname'),
 ('optdata', '&filepath.\\data\\optionalsdata'),
 ('othernew',
  '\\\\filestorage\\xstream\\cie\\csd\\ard_ild_reports\\&sessname\\peversion'),
 ('ref_xl', '&filepath\\data\\ild_config_tables_s&&sessionid&i...xlsx'),
 ('refdata', '&filepath.\\data'),
 ('refdata', '&filepath.\\data\\optionalsdata')}

In [361]:
all_tables

{('db_ca', 'AwardScheduleEvents'),
 ('db_ca', 'CandidateComponentMarkSet'),
 ('db_ca', 'QuestionPaperItemGroups'),
 ('db_ca', 'vAggCandCMS'),
 ('db_qp', 'AssessmentComponents'),
 ('db_qp', 'ItemTags'),
 ('db_qp', 'MarkSchemeItemTags'),
 ('db_qp', 'MarkingClusters'),
 ('db_r', 'SessAggCandCmpItemMarkIDs_S'),
 ('db_r', 'SessAggQPIGIDs_S'),
 ('db_r', 'SessAssCmpItemMarks_S'),
 ('db_s', 'SessionAssessmentComponents'),
 ('db_s', 'Sessions'),
 ('db_s', 'sessions'),
 ('db_sd', 'BusinessStreams'),
 ('optdata', 'componenttype11_s'),
 ('refdata', 'ComponentCalculationControl'),
 ('refdata', 'ComponentType'),
 ('refdata', 'ComponentsWithoutNewMarks'),
 ('refdata', 'Excluded_AssessmentComponents'),
 ('refdata', 'componenttype1_s'),
 ('refdata', 'excluded_MarkSchemes_S'),
 ('refdata', 'excluded_QuestionPapers'),
 ('refdata', 'excluded_candidates_S'),
 ('refdata', 'included_sessions'),
 ('refdata', 'no_ild')}

In [246]:
tbl_name

[('db_s', 'SessionAssessmentComponents')]

In [ ]:
###################################################################

In [1]:
import re

In [ ]:
def extract_sql_blocks(sas_script):
    # Extract SQL blocks (between PROC SQL; and QUIT;)
    pattern = re.compile(r'proc\s+sql.*?quit;', re.IGNORECASE | re.DOTALL)
    sql_keywords = {'quit', 'where', 'group', 'order', 'having', 'join', 'on', 'inner', 'left', 'right', 'full', 'outer', 'cross', 'union', 'except', 'intersect'}
    return pattern.findall(sas_script)

def extract_table_aliases(sql_block):
    alias_pattern = re.compile(r'from\s+([^\s]+)\s+(?:as\s+)?([^\s,]+)', re.IGNORECASE)
    sql_keywords = {'quit;','quit', 'where', 'group', 'order', 'having', 'join', 'on', 'inner', 'left', 'right', 'full', 'outer', 'cross', 'union', 'except', 'intersect'}
    alias_map = {}
    for table, alias in alias_pattern.findall(sql_block):
        if alias.lower() not in sql_keywords:
            alias_map[table] = alias
    return alias_map

In [45]:
with open(ild_scripts_path+r"\ild_01_identify_assessmentcomponents_requiring_recalculation.sas", "r", encoding="utf-8") as f:
        sas_script=f.read()
        sql_blocks =extract_sql_blocks(sas_script)
        for block in sql_blocks:
                alias_map = extract_table_aliases(block)
                break

In [ ]:
#####################################################################

{}